# CELL 1

In [25]:
%pip install pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = SparkSession.builder \
    .appName("capstone") \
    .master("local[*]") \
    .getOrCreate()

print("Cores available:", spark.sparkContext.defaultParallelism)


Note: you may need to restart the kernel to use updated packages.
Cores available: 8



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# My Machine: 8 crores

# Cell 2

In [21]:
orders_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("amount", DoubleType(), True),
    StructField("ts", StringType(), True),
    StructField("country", StringType(), True),
])

orders_raw = spark.read.csv("data/orders.csv", header=True, schema=orders_schema)
customers_raw = spark.read.csv("data/customers.csv", header=True, inferSchema=True)
products = spark.read.csv("data/products.csv", header=True, inferSchema=True)

orders_raw.printSchema()
orders_raw.show(5)

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- ts: string (nullable = true)
 |-- country: string (nullable = true)

+--------+-----------+----------+------+-------------------+-------+
|order_id|customer_id|product_id|amount|                 ts|country|
+--------+-----------+----------+------+-------------------+-------+
|       1|       9750|       259|144.97|2024-12-02 15:28:59|     GB|
|       2|       9622|       359| 83.38|2024-11-26 02:40:48|     AU|
|       3|       1260|       303| 24.75|2024-07-06 22:28:43|     NP|
|       4|       5420|        77| 25.15|2024-01-13 06:03:50|     NP|
|       5|       9753|       431| 27.11|2023-05-09 22:52:00|     NP|
+--------+-----------+----------+------+-------------------+-------+
only showing top 5 rows


# Part A: Clean the Data

In [22]:
start_count = orders_raw.count()
print("Starting order rows:", start_count)

Starting order rows: 1020025


In [23]:
# 1
orders_cleaned = orders_raw.dropDuplicates(['order_id'])    # Duplicates on order_id only because order_id is a unique business key

after_cleaned_count = orders_cleaned.count()
duplicates_removed = start_count - after_cleaned_count
print("Duplicate rows removed:", duplicates_removed)

Duplicate rows removed: 20025


In [ ]:
# Checks the null value or value that is 0 or negative and counts
bad_amount_count = orders_cleaned.filter(
    (F.col("amount").isNull()) | (F.col("amount") <= 0)
).count()
print("Bad amount rows (missing, zero, or negative):", bad_amount_count)

orders_amount_clean = orders_cleaned.filter(
    (F.col("amount").isNotNull()) & (F.col("amount") > 0)
)

Bad amount rows (missing, zero, or negative): 15022
